In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import json
import time
import os
import pickle

def setup_driver():
    """Set up a Chrome driver with enhanced bot detection evasion."""
    chrome_options = Options()
    # Headless mode disabled for debugging
    # chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.execute_cdp_cmd("Network.setUserAgentOverride", {
        "userAgent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    })
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def normalize_champion_name(champ):
    """Normalize champion names to match LoLalytics URL format (lowercase, no spaces or special characters)."""
    champ = champ.lower().replace(" ", "").replace("'", "").replace(".", "")
    special_cases = {
        "nunu&willump": "nunu",
        "drmundo": "drmundo",
        "ksante": "ksante",
        "aurelionsol": "aurelionsol",
        "jarvaniv": "jarvaniv",
        "leblanc": "leblanc",
        "leesin": "leesin",
        "twistedfate": "twistedfate",
        "velkoz": "velkoz",
        "tahmkench": "tahmkench",
        "belveth": "belveth",
        "renataglasc": "renataglasc"
    }
    return special_cases.get(champ, champ)

def scrape_champion_data(driver, champion, max_retries=3):
    """Scrape ADC matchup (Weak Against) and good synergy data from Common Matchups and Synergies tab."""
    normalized_champ = normalize_champion_name(champion)
    url = f"https://lolalytics.com/lol/{normalized_champ}/build/"
    for attempt in range(max_retries):
        try:
            driver.get(url)
            # Wait for the page to load
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div[id='root'], body"))
            )
            # Save initial page source
            with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_initial.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            print(f"Saved initial page source for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_initial.html")
            # Save screenshot
            driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_initial.png")
            print(f"Saved initial screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_initial.png")

            data = {"champion": champion, "matchups": [], "synergies": []}

            # Click the "Common Matchups and Synergies" tab
            try:
                tab_button = WebDriverWait(driver, 15).until(
                    EC.element_to_be_clickable((By.XPATH, "//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'matchups') or contains(@class, 'tab') or contains(@class, 'matchup')]"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", tab_button)
                driver.execute_script("arguments[0].click();", tab_button)
                print(f"Clicked 'Common Matchups and Synergies' tab for {champion}")
                time.sleep(5)  # Increased delay
                # Save page source and screenshot
                with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_tab.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"Saved page source after tab click for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_tab.html")
                driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_tab.png")
                print(f"Saved tab screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_tab.png")
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error clicking 'Common Matchups and Synergies' tab for {champion}: {e}")

            # Click the "Weak Against" button for matchups
            try:
                weak_button = WebDriverWait(driver, 15).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-type='weak_counter']"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", weak_button)
                driver.execute_script("arguments[0].click();", weak_button)
                print(f"Clicked 'Weak Against' button for {champion}")
                time.sleep(5)  # Increased delay
                # Save page source and screenshot
                with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_weak.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"Saved page source after weak button click for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_weak.html")
                driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_weak.png")
                print(f"Saved weak button screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_weak.png")
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error clicking 'Weak Against' button for {champion}: {e}")

            # Scrape matchups (Weak Against filter)
            try:
                matchup_table = WebDriverWait(driver, 30).until(
                    EC.visibility_of_element_located((By.CSS_SELECTOR, "table[class*='matchup'], div[class*='matchup'] table"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", matchup_table)
                rows = matchup_table.find_elements(By.TAG_NAME, "tr")
                print(f"Found {len(rows)} rows in matchup table for {champion}")
                for row in rows[1:]:  # Skip header row
                    cols = row.find_elements(By_TAG_NAME, "td")
                    if len(cols) >= 3:
                        matchup_data = {
                            "opponent": cols[0].text.strip(),
                            "win_rate": cols[1].text.strip(),
                            "games": cols[2].text.strip()
                        }
                        data["matchups"].append(matchup_data)
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error scraping matchups for {champion}: {e}")
                # Log table HTML if found
                try:
                    tables = driver.find_elements(By.CSS_SELECTOR, "table[class*='matchup'], div[class*='matchup'] table")
                    for i, table in enumerate(tables):
                        print(f"Matchup table {i} HTML: {table.get_attribute('outerHTML')[:200]}...")
                except:
                    print("No matchup tables found for logging.")

            # Click the "Good Synergy" button for synergies
            try:
                synergy_button = WebDriverWait(driver, 15).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-type='good_synergy']"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", synergy_button)
                driver.execute_script("arguments[0].click();", synergy_button)
                print(f"Clicked 'Good Synergy' button for {champion}")
                time.sleep(5)  # Increased delay
                # Save page source and screenshot
                with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_synergy.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"Saved page source after synergy click for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_synergy.html")
                driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_synergy.png")
                print(f"Saved synergy screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_synergy.png")
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error clicking 'Good Synergy' button for {champion}: {e}")

            # Scrape synergies (Good Synergy filter)
            try:
                synergy_table = WebDriverWait(driver, 30).until(
                    EC.visibility_of_element_located((By.CSS_SELECTOR, "table[class*='synergy'], div[class*='synergy'] table"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", synergy_table)
                rows = synergy_table.find_elements(By_TAG_NAME, "tr")
                print(f"Found {len(rows)} rows in synergy table for {champion}")
                for row in rows[1:]:  # Skip header row
                    cols = row.find_elements(By_TAG_NAME, "td")
                    if len(cols) >= 3:
                        synergy_data = {
                            "ally": cols[0].text.strip(),
                            "win_rate": cols[1].text.strip(),
                            "games": cols[2].text.strip()
                        }
                        data["synergies"].append(synergy_data)
            except Exception as e:
                print(f"Attempt {attempt + 1}: Error scraping synergies for {champion}: {e}")
                # Log table HTML if found
                try:
                    tables = driver.find_elements(By.CSS_SELECTOR, "table[class*='synergy'], div[class*='synergy'] table")
                    for i, table in enumerate(tables):
                        print(f"Synergy table {i} HTML: {table.get_attribute('outerHTML')[:200]}...")
                except:
                    print("No synergy tables found for logging.")

            if data["matchups"] or data["synergies"]:
                return data
            print(f"Attempt {attempt + 1}: No data scraped for {champion}, retrying...")
            time.sleep(5)

        except Exception as e:
            print(f"Attempt {attempt + 1}: Error loading page for {champion}: {e}")
            with open(f"debug_page_source_{champion}_attempt_{attempt + 1}_error.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            print(f"Saved page source for {champion} to debug_page_source_{champion}_attempt_{attempt + 1}_error.html")
            driver.save_screenshot(f"debug_screenshot_{champion}_attempt_{attempt + 1}_error.png")
            print(f"Saved error screenshot for {champion} to debug_screenshot_{champion}_attempt_{attempt + 1}_error.png")
            time.sleep(5)

    print(f"Failed to scrape data for {champion} after {max_retries} attempts.")
    return {"champion": champion, "matchups": [], "synergies": []}

def main():
    driver = setup_driver()
    try:
        # Load champions from pickle file
        try:
            with open("champions.pkl", "rb") as champion_file:
                champions = pickle.load(champion_file)
        except FileNotFoundError:
            print("Error: champions.pkl not found. Exiting.")
            return
        except pickle.PickleError as e:
            print(f"Error: Failed to load champions.pkl: {e}. Exiting.")
            return

        # Validate champion list
        champions = list(dict.fromkeys(champions))  # Remove duplicates
        if not champions:
            print("No champions found. Exiting.")
            return
        print(f"Found {len(champions)} champions: {champions}")

        # Check for unexpected champion count
        expected_count = 171  # Updated to expect 171 champions
        if len(champions) != expected_count:
            print(f"Warning: Expected {expected_count} champions, but found {len(champions)}. Please verify champions.pkl.")

        all_data = []
        total_champions = len(champions)
        for i, champion in enumerate(champions, 1):
            print(f"Scraping data for {champion} ({i}/{total_champions})...")
            champ_data = scrape_champion_data(driver, champion)
            all_data.append(champ_data)
            # Save partial data every 10 champions
            if i % 10 == 0:
                with open("lolalytics_champion_data_partial.json", "w", encoding="utf-8") as f:
                    json.dump(all_data, f, indent=2)
                print(f"Partial data saved to lolalytics_champion_data_partial.json after {i} champions")
            time.sleep(3)

        # Save final data
        with open("lolalytics_champion_data.json", "w", encoding="utf-8") as f:
            json.dump(all_data, f, indent=2)
        print("Data saved to lolalytics_champion_data.json")

    finally:
        driver.quit()

if __name__ == "__main__":
    main()

Found 171 champions: ['aatrox', 'ahri', 'akali', 'akshan', 'alistar', 'ambessa', 'amumu', 'anivia', 'annie', 'aphelios', 'ashe', 'aurelionsol', 'aurora', 'azir', 'bard', 'belveth', 'blitzcrank', 'brand', 'braum', 'briar', 'caitlyn', 'camille', 'cassiopeia', 'chogath', 'corki', 'darius', 'diana', 'drmundo', 'draven', 'ekko', 'elise', 'evelynn', 'ezreal', 'fiddlesticks', 'fiora', 'fizz', 'galio', 'gangplank', 'garen', 'gnar', 'gragas', 'graves', 'gwen', 'hecarim', 'heimerdinger', 'hwei', 'illaoi', 'irelia', 'ivern', 'janna', 'jarvaniv', 'jax', 'jayce', 'jhin', 'jinx', 'ksante', 'kaisa', 'kalista', 'karma', 'karthus', 'kassadin', 'katarina', 'kayle', 'kayn', 'kennen', 'khazix', 'kindred', 'kled', 'kogmaw', 'leblanc', 'leesin', 'leona', 'lillia', 'lissandra', 'lucian', 'lulu', 'lux', 'malphite', 'malzahar', 'maokai', 'masteryi', 'mel', 'milio', 'missfortune', 'mordekaiser', 'morgana', 'naafiri', 'nami', 'nasus', 'nautilus', 'neeko', 'nidalee', 'nilah', 'nocturne', 'nunu&willump', 'olaf', '

KeyboardInterrupt: 

In [13]:
import pickle
champions = [
    "Aatrox", "Ahri", "Akali", "Akshan", "Alistar", "Ambessa", "Amumu", "Anivia", "Annie", "Aphelios",
    "Ashe", "Aurelion Sol", "Aurora", "Azir", "Bard", "Belveth", "Blitzcrank", "Brand", "Braum", "Briar",
    "Caitlyn", "Camille", "Cassiopeia", "Chogath", "Corki", "Darius", "Diana", "Dr. Mundo", "Draven", "Ekko",
    "Elise", "Evelynn", "Ezreal", "Fiddlesticks", "Fiora", "Fizz", "Galio", "Gangplank", "Garen", "Gnar",
    "Gragas", "Graves", "Gwen", "Hecarim", "Heimerdinger", "Hwei", "Illaoi", "Irelia", "Ivern", "Janna",
    "Jarvan IV", "Jax", "Jayce", "Jhin", "Jinx", "K'Sante", "Kai'Sa", "Kalista", "Karma", "Karthus",
    "Kassadin", "Katarina", "Kayle", "Kayn", "Kennen", "Kha'Zix", "Kindred", "Kled", "Kog'Maw", "LeBlanc",
    "Lee Sin", "Leona", "Lillia", "Lissandra", "Lucian", "Lulu", "Lux", "Malphite", "Malzahar", "Maokai",
    "Master Yi", "Mel", "Milio", "Miss Fortune", "Mordekaiser", "Morgana", "Naafiri", "Nami", "Nasus",
    "Nautilus", "Neeko", "Nidalee", "Nilah", "Nocturne", "Nunu & Willump", "Olaf", "Orianna", "Ornn",
    "Pantheon", "Poppy", "Pyke", "Qiyana", "Quinn", "Rakan", "Rammus", "Rek'Sai", "Rell", "Renata Glasc",
    "Renekton", "Rengar", "Riven", "Rumble", "Ryze", "Samira", "Sejuani", "Senna", "Seraphine", "Sett",
    "Shaco", "Shen", "Shyvana", "Singed", "Sion", "Sivir", "Skarner", "Smolder", "Sona", "Soraka",
    "Swain", "Sylas", "Syndra", "Tahm Kench", "Taliyah", "Talon", "Taric", "Teemo", "Thresh", "Tristana",
    "Trundle", "Tryndamere", "Twisted Fate", "Twitch", "Udyr", "Urgot", "Varus", "Vayne", "Veigar",
    "Vel'Koz", "Vex", "Vi", "Viego", "Viktor", "Vladimir", "Volibear", "Warwick", "Wukong", "Xayah",
    "Xerath", "Xin Zhao", "Yasuo", "Yone", "Yorick", "Yunara", "Yuumi", "Zaahen", "Zac", "Zed", "Zeri", "Ziggs",
    "Zilean", "Zoe", "Zyra", 
]
champions = [champion.lower() for champion in champions]
champions = [champion.replace(" ", "") for champion in champions]
champions = [champion.replace("'", "") for champion in champions]
champions = [champion.replace(".", "") for champion in champions]
print(champions)
with open("champions.pkl", "wb") as champion_file:
    pickle.dump(champions, champion_file)
    

['aatrox', 'ahri', 'akali', 'akshan', 'alistar', 'ambessa', 'amumu', 'anivia', 'annie', 'aphelios', 'ashe', 'aurelionsol', 'aurora', 'azir', 'bard', 'belveth', 'blitzcrank', 'brand', 'braum', 'briar', 'caitlyn', 'camille', 'cassiopeia', 'chogath', 'corki', 'darius', 'diana', 'drmundo', 'draven', 'ekko', 'elise', 'evelynn', 'ezreal', 'fiddlesticks', 'fiora', 'fizz', 'galio', 'gangplank', 'garen', 'gnar', 'gragas', 'graves', 'gwen', 'hecarim', 'heimerdinger', 'hwei', 'illaoi', 'irelia', 'ivern', 'janna', 'jarvaniv', 'jax', 'jayce', 'jhin', 'jinx', 'ksante', 'kaisa', 'kalista', 'karma', 'karthus', 'kassadin', 'katarina', 'kayle', 'kayn', 'kennen', 'khazix', 'kindred', 'kled', 'kogmaw', 'leblanc', 'leesin', 'leona', 'lillia', 'lissandra', 'lucian', 'lulu', 'lux', 'malphite', 'malzahar', 'maokai', 'masteryi', 'mel', 'milio', 'missfortune', 'mordekaiser', 'morgana', 'naafiri', 'nami', 'nasus', 'nautilus', 'neeko', 'nidalee', 'nilah', 'nocturne', 'nunu&willump', 'olaf', 'orianna', 'ornn', 'pa

In [7]:
with open("champions.pkl", "rb") as champion_file:
    champions = pickle.load(champion_file)
print(champions)

['aatrox', 'ahri', 'akali', 'akshan', 'alistar', 'ambessa', 'amumu', 'anivia', 'annie', 'aphelios', 'ashe', 'aurelionsol', 'aurora', 'azir', 'bard', 'belveth', 'blitzcrank', 'brand', 'braum', 'briar', 'caitlyn', 'camille', 'cassiopeia', 'chogath', 'corki', 'darius', 'diana', 'drmundo', 'draven', 'ekko', 'elise', 'evelynn', 'ezreal', 'fiddlesticks', 'fiora', 'fizz', 'galio', 'gangplank', 'garen', 'gnar', 'gragas', 'graves', 'gwen', 'hecarim', 'heimerdinger', 'hwei', 'illaoi', 'irelia', 'ivern', 'janna', 'jarvaniv', 'jax', 'jayce', 'jhin', 'jinx', 'ksante', 'kaisa', 'kalista', 'karma', 'karthus', 'kassadin', 'katarina', 'kayle', 'kayn', 'kennen', 'khazix', 'kindred', 'kled', 'kogmaw', 'leblanc', 'leesin', 'leona', 'lillia', 'lissandra', 'lucian', 'lulu', 'lux', 'malphite', 'malzahar', 'maokai', 'masteryi', 'mel', 'milio', 'missfortune', 'mordekaiser', 'morgana', 'naafiri', 'nami', 'nasus', 'nautilus', 'neeko', 'nidalee', 'nilah', 'nocturne', 'nunu&willump', 'olaf', 'orianna', 'ornn', 'pa

In [12]:
import argparse
import json
import re
import time
from dataclasses import dataclass, asdict
from typing import Dict, Iterable, List, Optional, Tuple

from selenium import webdriver
from selenium.webdriver import ChromeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service


# -----------------------------
# Champion list (paste yours)
# -----------------------------
champions = [
    "Aatrox", "Ahri", "Akali", "Akshan", "Alistar", "Ambessa", "Amumu", "Anivia", "Annie", "Aphelios",
    "Ashe", "Aurelion Sol", "Aurora", "Azir", "Bard", "Belveth", "Blitzcrank", "Brand", "Braum", "Briar",
    "Caitlyn", "Camille", "Cassiopeia", "Chogath", "Corki", "Darius", "Diana", "Dr. Mundo", "Draven", "Ekko",
    "Elise", "Evelynn", "Ezreal", "Fiddlesticks", "Fiora", "Fizz", "Galio", "Gangplank", "Garen", "Gnar",
    "Gragas", "Graves", "Gwen", "Hecarim", "Heimerdinger", "Hwei", "Illaoi", "Irelia", "Ivern", "Janna",
    "Jarvan IV", "Jax", "Jayce", "Jhin", "Jinx", "K'Sante", "Kai'Sa", "Kalista", "Karma", "Karthus",
    "Kassadin", "Katarina", "Kayle", "Kayn", "Kennen", "Kha'Zix", "Kindred", "Kled", "Kog'Maw", "LeBlanc",
    "Lee Sin", "Leona", "Lillia", "Lissandra", "Lucian", "Lulu", "Lux", "Malphite", "Malzahar", "Maokai",
    "Master Yi", "Mel", "Milio", "Miss Fortune", "Mordekaiser", "Morgana", "Naafiri", "Nami", "Nasus",
    "Nautilus", "Neeko", "Nidalee", "Nilah", "Nocturne", "Nunu & Willump", "Olaf", "Orianna", "Ornn",
    "Pantheon", "Poppy", "Pyke", "Qiyana", "Quinn", "Rakan", "Rammus", "Rek'Sai", "Rell", "Renata Glasc",
    "Renekton", "Rengar", "Riven", "Rumble", "Ryze", "Samira", "Sejuani", "Senna", "Seraphine", "Sett",
    "Shaco", "Shen", "Shyvana", "Singed", "Sion", "Sivir", "Skarner", "Smolder", "Sona", "Soraka",
    "Swain", "Sylas", "Syndra", "Tahm Kench", "Taliyah", "Talon", "Taric", "Teemo", "Thresh", "Tristana",
    "Trundle", "Tryndamere", "Twisted Fate", "Twitch", "Udyr", "Urgot", "Varus", "Vayne", "Veigar",
    "Vel'Koz", "Vex", "Vi", "Viego", "Viktor", "Vladimir", "Volibear", "Warwick", "Wukong", "Xayah",
    "Xerath", "Xin Zhao", "Yasuo", "Yone", "Yorick", "Yunara", "Yuumi","Zaahen", "Zac", "Zed", "Zeri", "Ziggs",
    "Zilean", "Zoe", "Zyra"
]


In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://lolalytics.com/lol/ashe/build/"

response = requests.get(url)
response.raise_for_status()  # optional but recommended

soup = BeautifulSoup(response.text, "html.parser")


In [1]:
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time
import pandas as pd
from typing import List, Dict, Any, Tuple
import undetected_chromedriver as uc

# --- Configuration ---
CHAMPIONS_FILE = 'champions.pkl'
BASE_URL = 'https://lolalytics.com/lol/{}/build'
OUTPUT_FILE = 'lolalytics_draft_data_combined.csv'
MAX_MATCHUPS_TO_SCRAPE = 50

# --- Retry / speed settings ---
MAX_CHAMP_RETRIES = 3
RETRY_COOLDOWN_S = 4
RESTART_DRIVER_EVERY = 25
SCROLL_PAUSE_S = 0.3
MAX_SCROLL_ROUNDS = 40
STABLE_ROUNDS_TO_STOP = 3   # stop when no NEW champs appear for this many rounds
TEMP_SAVE_EVERY = 5

SUPPORT_CHAMPS = {
    'Alistar', 'Bard', 'Brand', 'Blitzcrank', 'Braum', 'Janna', 'Karma', 'Leona', 'Lulu',
    'Lux', 'Maokai', 'Milio', 'Morgana', 'Nami', 'Nautilus', "Neeko", "Poppy", 'Pantheon', 'Pyke',
    'Rakan', 'Rell', 'Renata', 'Senna', 'Seraphine',
    'Sona', 'Soraka', 'Swain', 'Tahmkench', 'Taric', 'Thresh', "Velkoz", 'Yuumi',
    'Zilean', 'Zyra'
}
SUPPORT_CHAMPS = set(c.lower() for c in SUPPORT_CHAMPS)

# --- XPATHS ---
COOKIE_ACCEPT_BUTTON_XPATH = "//button[contains(text(), 'Accept')]"
WEAK_MATCHUPS_TAB_XPATH = "/html/body/main/div[6]/div[1]/div[1]/div[1]/div[2]/div/div[3]"
GOOD_SYNERGIES_TAB_XPATH = "/html/body/main/div[6]/div[1]/div[1]/div[2]/div[2]/div[2]"

SYNERGY_DATA_BASE_XPATH = "/html/body/main/div[6]/div[1]/div[4]/div[2]/div/div"   # div[4] non-support
MATCHUP_DATA_BASE_XPATH = "/html/body/main/div[6]/div[1]/div[5]/div[2]/div/div"   # div[5] matchups + support synergies

BUILD_SUMMARY_XPATH = "/html/body/main/div[6]/div[1]/div[2]"
THIRD_MATCHUP_CHAMP_XPATH = f"{MATCHUP_DATA_BASE_XPATH}[3]/a/span/img"

# --- JS: extract ALL currently rendered rows from the list container ---
# The list is virtualized: rows scrolled out of view are removed from the DOM.
# So we iterate the container's live children (no fixed indices, no early break)
# and let Python accumulate results across scroll rounds.
JS_EXTRACT_RENDERED = """
const base = arguments[0];
// base looks like ".../div[2]/div/div" (row pattern) -> container is its parent
const containerXp = base.substring(0, base.lastIndexOf('/'));
const container = document.evaluate(containerXp, document, null,
    XPathResult.FIRST_ORDERED_NODE_TYPE, null).singleNodeValue;
const out = [];
if (!container) return out;
for (const row of container.children) {
    const img = document.evaluate('./a/span/img', row, null,
        XPathResult.FIRST_ORDERED_NODE_TYPE, null).singleNodeValue;
    const val = document.evaluate('./div[1]/span', row, null,
        XPathResult.FIRST_ORDERED_NODE_TYPE, null).singleNodeValue;
    if (!img || !val) continue;
    const alt = img.getAttribute('alt');
    const txt = val.textContent.trim();
    if (alt && txt) out.push([alt, txt]);
}
return out;
"""

# Scroll the container itself (works for horizontal and vertical virtual lists)
JS_SCROLL_CONTAINER = """
const base = arguments[0];
const containerXp = base.substring(0, base.lastIndexOf('/'));
const container = document.evaluate(containerXp, document, null,
    XPathResult.FIRST_ORDERED_NODE_TYPE, null).singleNodeValue;
if (!container) return false;
// Prefer scrolling the last rendered row into view, then nudge the scrollable
// ancestor a bit further to force the virtualizer to mount the next batch.
const last = container.lastElementChild;
if (last) last.scrollIntoView({block: 'nearest', inline: 'end'});
let el = container;
while (el && el !== document.body) {
    const canX = el.scrollWidth  > el.clientWidth;
    const canY = el.scrollHeight > el.clientHeight;
    if (canX) { el.scrollLeft += el.clientWidth  * 0.8; return true; }
    if (canY) { el.scrollTop  += el.clientHeight * 0.8; return true; }
    el = el.parentElement;
}
return true;
"""


def setup_driver():
    options = uc.ChromeOptions()
    options.add_argument('--log-level=3')
    options.add_argument('--blink-settings=imagesDisabled=true')
    options.add_argument('--disable-gpu')
    options.page_load_strategy = 'eager'
    driver = uc.Chrome(options=options, version_main=148, use_subprocess=True)
    return driver


def load_champions(filename: str) -> List[str]:
    try:
        with open(filename, 'rb') as f:
            return [str(c) for c in pickle.load(f)]
    except FileNotFoundError:
        print(f"Error: Champion file '{filename}' not found. Please create it first.")
        return []
    except Exception as e:
        print(f"An error occurred loading champions: {e}")
        return []


def format_champion_name(name: str) -> str:
    return name.lower().replace(" ", "").replace("'", "")


def count_categories(data: List[Dict[str, Any]]) -> Tuple[int, int]:
    m = sum(1 for d in data if d.get('category') == 'WEAK_MATCHUP')
    s = sum(1 for d in data if d.get('category') == 'GOOD_SYNERGY')
    return m, s


def harvest_rows(driver, base_xpath: str, max_rows: int) -> List[Tuple[str, str]]:
    """
    Accumulates rows across scroll rounds. Because the list is virtualized,
    every round we read whatever is rendered RIGHT NOW and merge it into
    `collected` (first value wins, insertion order preserved). Scrolling
    continues until no new champion names appear for STABLE_ROUNDS_TO_STOP
    rounds or max_rows is reached.
    """
    collected: Dict[str, str] = {}
    stable_rounds = 0

    for _ in range(MAX_SCROLL_ROUNDS):
        rendered = driver.execute_script(JS_EXTRACT_RENDERED, base_xpath)

        new_found = 0
        for name, value in rendered:
            if name not in collected:
                collected[name] = value
                new_found += 1

        if len(collected) >= max_rows:
            break

        if new_found == 0:
            stable_rounds += 1
            if stable_rounds >= STABLE_ROUNDS_TO_STOP:
                break
        else:
            stable_rounds = 0

        driver.execute_script(JS_SCROLL_CONTAINER, base_xpath)
        time.sleep(SCROLL_PAUSE_S)

    return list(collected.items())[:max_rows]


def scrape_data(driver, champion: str) -> List[Dict[str, Any]]:
    formatted_champ = format_champion_name(champion)
    url = BASE_URL.format(formatted_champ)
    print(f"\n--- Scraping: {champion} (Max {MAX_MATCHUPS_TO_SCRAPE} each) ---")

    driver.get(url)
    wait = WebDriverWait(driver, 15)
    champion_data: List[Dict[str, Any]] = []

    # 1. Initial page load + cookie banner
    try:
        wait.until(EC.presence_of_element_located((By.XPATH, BUILD_SUMMARY_XPATH)))
        try:
            cookie_button = WebDriverWait(driver, 2).until(
                EC.element_to_be_clickable((By.XPATH, COOKIE_ACCEPT_BUTTON_XPATH)))
            driver.execute_script("arguments[0].click();", cookie_button)
        except Exception:
            pass
    except Exception as e:
        print(f" -> ERROR: Initial page content failed to load. {e}")
        return []

    driver.execute_script("window.scrollTo(0, 1500);")

    # --- BLOCK 1: WEAK MATCHUPS (div[5]) ---
    print("[WEAK MATCHUPS]")
    matchup_success = False
    for attempt in (1, 2):
        try:
            tab = wait.until(EC.presence_of_element_located((By.XPATH, WEAK_MATCHUPS_TAB_XPATH)))
            driver.execute_script("arguments[0].click();", tab)
            time.sleep(0.5)
            WebDriverWait(driver, 6).until(
                EC.presence_of_element_located((By.XPATH, THIRD_MATCHUP_CHAMP_XPATH)))
            matchup_success = True
            break
        except Exception:
            if attempt == 1:
                print(" -> Matchup rendering failed. Retrying tab click...")
                time.sleep(1)
            else:
                print(" -> Matchup rendering failed after all attempts. Skipping.")

    if matchup_success:
        rows = harvest_rows(driver, MATCHUP_DATA_BASE_XPATH, MAX_MATCHUPS_TO_SCRAPE)
        for vs_champ, value in rows:
            champion_data.append({'champion': champion, 'category': 'WEAK_MATCHUP',
                                  'vs_champ': vs_champ, 'value': value})
        print(f" -> Extracted {len(rows)} weak matchup rows.")

    # --- BLOCK 2: GOOD SYNERGIES ---
    print("[GOOD SYNERGIES]")
    try:
        tab = wait.until(EC.presence_of_element_located((By.XPATH, GOOD_SYNERGIES_TAB_XPATH)))
        driver.execute_script("arguments[0].click();", tab)
        time.sleep(0.5)

        if format_champion_name(champion) in SUPPORT_CHAMPS:
            synergy_base_path = MATCHUP_DATA_BASE_XPATH
            print(f" -> {champion} is a SUPPORT: using div[5].")
        else:
            synergy_base_path = SYNERGY_DATA_BASE_XPATH
            print(f" -> {champion} is NON-SUPPORT: using div[4].")

        try:
            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, f"{synergy_base_path}[1]/a")))
        except TimeoutException:
            print(" -> WARNING: Synergy data failed initial load check.")

        rows = harvest_rows(driver, synergy_base_path, MAX_MATCHUPS_TO_SCRAPE)
        for with_champ, value in rows:
            champion_data.append({'champion': champion, 'category': 'GOOD_SYNERGY',
                                  'with_champ': with_champ, 'value': value})
        print(f" -> Extracted {len(rows)} synergy rows.")

    except Exception as e:
        print(f" -> ERROR: Good Synergy scrape failed: {e}")

    return champion_data


class DriverPool:
    """Keeps one browser alive across champions; recreates it only when needed."""

    def __init__(self):
        self.driver = None
        self.champs_served = 0

    def get(self, force_new: bool = False):
        if force_new or self.driver is None or self.champs_served >= RESTART_DRIVER_EVERY:
            self.close()
            print(" -> Starting fresh Chrome instance...")
            self.driver = setup_driver()
            self.champs_served = 0
        self.champs_served += 1
        return self.driver

    def close(self):
        if self.driver is not None:
            try:
                self.driver.quit()
            except Exception:
                pass
            self.driver = None


def scrape_champion_with_retries(pool: DriverPool, champion: str) -> List[Dict[str, Any]]:
    best_data: List[Dict[str, Any]] = []

    for attempt in range(1, MAX_CHAMP_RETRIES + 1):
        force_new = attempt > 1
        if force_new:
            print(f"\n*** RETRY {attempt}/{MAX_CHAMP_RETRIES} for {champion} ***")
            time.sleep(RETRY_COOLDOWN_S)

        data: List[Dict[str, Any]] = []
        try:
            driver = pool.get(force_new=force_new)
            data = scrape_data(driver, champion)
        except Exception as e:
            print(f" -> ERROR during attempt {attempt} for {champion}: {e}")
            pool.close()

        m, s = count_categories(data)
        print(f" -> Attempt {attempt}: {m} matchups, {s} synergies.")

        if len(data) > len(best_data):
            best_data = data
        if m > 0 and s > 0:
            return data

    print(f" -> WARNING: {champion} still incomplete after {MAX_CHAMP_RETRIES} attempts "
          f"(keeping best attempt, {len(best_data)} rows).")
    return best_data


def run_scraper():
    champions = load_champions(CHAMPIONS_FILE)
    if not champions:
        return

    all_data: List[Dict[str, Any]] = []
    failed_champs: List[str] = []
    pool = DriverPool()
    t0 = time.time()

    try:
        for idx, champ in enumerate(champions, 1):
            data = scrape_champion_with_retries(pool, champ)
            m, s = count_categories(data)
            if m == 0 or s == 0:
                failed_champs.append(champ)
            all_data.extend(data)

            if idx % TEMP_SAVE_EVERY == 0 or idx == len(champions):
                pd.DataFrame(all_data).to_csv("lolalytics_data_percent_temp.csv", index=False)

            elapsed = time.time() - t0
            print(f" -> Progress: {idx}/{len(champions)} champs, "
                  f"{elapsed/idx:.1f}s avg/champ, {elapsed/60:.1f} min elapsed.")
    except Exception as e:
        print(f"An unexpected error occurred in the main loop: {e}")
    finally:
        pool.close()

    if all_data:
        df = pd.DataFrame(all_data)
        df.to_csv(OUTPUT_FILE, index=False)
        df.to_excel("lolalytics_data_percent.xlsx")
        print("\n--- Scraper Finished ---")
        print(f"Data scraped for {len(df['champion'].unique())} champions, saved to {OUTPUT_FILE}")
        if failed_champs:
            print(f"\nWARNING: Still missing/incomplete after retries: {failed_champs}")
        print("\nExample Data:")
        print(df.head())
    else:
        print("WARNING: No data was successfully scraped.")


if __name__ == '__main__':
    run_scraper()

 -> Starting fresh Chrome instance...

--- Scraping: aatrox (Max 50 each) ---
[WEAK MATCHUPS]
 -> Extracted 33 weak matchup rows.
[GOOD SYNERGIES]
 -> aatrox is NON-SUPPORT: using div[4].
 -> Extracted 31 synergy rows.
 -> Attempt 1: 33 matchups, 31 synergies.
 -> Progress: 1/172 champs, 9.3s avg/champ, 0.2 min elapsed.

--- Scraping: ahri (Max 50 each) ---
[WEAK MATCHUPS]
 -> Extracted 30 weak matchup rows.
[GOOD SYNERGIES]
 -> ahri is NON-SUPPORT: using div[4].
 -> Extracted 30 synergy rows.
 -> Attempt 1: 30 matchups, 30 synergies.
 -> Progress: 2/172 champs, 8.5s avg/champ, 0.3 min elapsed.

--- Scraping: akali (Max 50 each) ---
[WEAK MATCHUPS]
 -> Extracted 33 weak matchup rows.
[GOOD SYNERGIES]
 -> akali is NON-SUPPORT: using div[4].
 -> Extracted 29 synergy rows.
 -> Attempt 1: 33 matchups, 29 synergies.
 -> Progress: 3/172 champs, 8.2s avg/champ, 0.4 min elapsed.

--- Scraping: akshan (Max 50 each) ---
[WEAK MATCHUPS]
 -> Extracted 33 weak matchup rows.
[GOOD SYNERGIES]
 -> aks

In [38]:
# to scrape the deltas
import pickle
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import pandas as pd
from typing import List, Dict, Any

# --- Configuration (Optimized settings remain the same) ---
CHAMPIONS_FILE = 'champions.pkl'
BASE_URL = 'https://lolalytics.com/lol/{}/build'
OUTPUT_FILE = 'lolalytics_draft_data_normalized.xlsx' # Changed output file name for clarity
MAX_MATCHUPS_TO_SCRAPE = 50
ROW_TIMEOUT = 3 

# --- SUPPORT CHAMPION LIST (Retained for synergy block logic) ---
SUPPORT_CHAMPS = {
    'Alistar', 'Bard', 'Brand', 'Blitzcrank', 'Braum', 'Janna', 'Karma', 'Leona', 'Lulu', 
    'Lux', 'Maokai', 'Milio', 'Morgana', 'Nami', 'Nautilus', "Neeko", "Poppy", 'Pantheon', 'Pyke', 
    'Rakan', 'Rell', 'Renata Glasc', 'Senna', 'Seraphine', 
    'Sona', 'Soraka', 'Swain', 'Tahm Kench', 'Taric', 'Thresh', "Velkoz", 'Yuumi', 
    'Zilean', 'Zyra'
}
SUPPORT_CHAMPS = set([champ.lower() for champ in SUPPORT_CHAMPS])

# --- UPDATED XPATHS ---
COOKIE_ACCEPT_BUTTON_XPATH = "//button[contains(text(), 'Accept')]" 
# 1. UPDATED: Normalized Matchup button
WEAK_MATCHUPS_TAB_XPATH = "/html/body/main/div[6]/div[1]/div[1]/div[1]/div[2]/div/div[4]"
# 2. UPDATED: Normalized Synergy button
GOOD_SYNERGIES_TAB_XPATH = "/html/body/main/div[6]/div[1]/div[1]/div[2]/div[2]/div[5]"
# /html/body/main/div[6]/div[1]/div[1]/div[2]/div[2]/div[4]
# /html/body/main/div[6]/div[1]/div[1]/div[2]/div[2]/div[5]

# DISTINCT BASE XPATHS (These relate to content *block* location, not the tab)
SYNERGY_DATA_BASE_XPATH = "/html/body/main/div[6]/div[1]/div[4]/div[2]/div/div" 
#                          /html/body/main/div[6]/div[1]/div[4]/div[2]/div/div[1]/div[2]
#                          /html/body/main/div[6]/div[1]/div[4]/div[2]/div/div[2]/div[2]
#                          /html/body/main/div[6]/div[1]/div[4]/div[2]/div/div[3]/div[2]
MATCHUP_DATA_BASE_XPATH = "/html/body/main/div[6]/div[1]/div[5]/div[2]/div/div" 

# Elements used for waiting/checking
BUILD_SUMMARY_XPATH = "/html/body/main/div[6]/div[1]/div[2]" 
THIRD_MATCHUP_CHAMP_XPATH = f"{MATCHUP_DATA_BASE_XPATH}[3]/a/span/img" # Image XPATH remains for stability check

# 3. UPDATED VALUE LOCATION XPATH (Both Matchup and Synergy numbers are now in div[3])
WIN_RATE_DIV_LOCATION = "3" 
# /html/body/main/div[6]/div[1]/div[5]/div[2]/div/div[1]/div[3]


# --- Helper Functions (setup_driver, load_champions, format_champion_name remain the same) ---
def setup_driver():
    """Initializes the Chrome WebDriver in VISIBLE MODE (for reliability)."""
    options = webdriver.ChromeOptions()
    options.add_argument('--log-level=3') 
    driver = webdriver.Chrome(service=Service(), options=options)
    return driver

def load_champions(filename: str) -> List[str]:
    """Loads the list of champions from a pickle file."""
    try:
        with open(filename, 'rb') as f:
            champs = pickle.load(f)
            return [str(c) for c in champs]
    except FileNotFoundError:
        print(f"Error: Champion file '{filename}' not found. Please create it first.")
        return []
    except Exception as e:
        print(f"An error occurred loading champions: {e}")
        return []

def format_champion_name(name: str) -> str:
    """Formats champion names for the Lolalytics URL."""
    return name.lower().replace(" ", "").replace("'", "")

def scrape_data(driver: webdriver.Chrome, champion: str) -> List[Dict[str, Any]]:
    """Navigates, clicks, scrolls, and scrapes both Weak Matchups and Good Synergies."""
    formatted_champ = format_champion_name(champion)
    url = BASE_URL.format(formatted_champ)
    print(f"\n--- Scraping Normalized Matchups and Synergies for: {champion} (Max {MAX_MATCHUPS_TO_SCRAPE} each) ---")
    
    driver.get(url)
    wait = WebDriverWait(driver, 20)
    champion_data = []

    # 1. INITIAL PAGE LOAD & COOKIE HANDLER (Same as before)
    print(" -> Waiting for core page content to load...")
    try:
        wait.until(EC.presence_of_element_located((By.XPATH, BUILD_SUMMARY_XPATH)))
        try:
            cookie_button = wait.until(EC.element_to_be_clickable((By.XPATH, COOKIE_ACCEPT_BUTTON_XPATH)))
            driver.execute_script("arguments[0].click();", cookie_button)
        except Exception:
            pass
    except Exception as e:
        print(f" -> ERROR: Initial page content or cookie banner failed to load quickly. Error: {e}")
        return []
    
    # 2. Initial Scroll Down (Same as before)
    print(" -> Scrolling down to load dynamic content area...")
    driver.execute_script("window.scrollTo(0, 1500);")
    
    # --- BLOCK 1: WEAK MATCHUPS (Normalized button click) ---
    print("\n[WEAK MATCHUPS - NORMALIZED]")
    max_attempts = 2 
    attempt = 0
    matchup_success = False

    while attempt < max_attempts and not matchup_success:
        attempt += 1
        # Click the NEW Normalized Matchups tab
        print(f" -> Attempt {attempt}: Clicking Normalized Matchups tab via JS...")
        try:
            weak_matchups_tab = wait.until(EC.presence_of_element_located((By.XPATH, WEAK_MATCHUPS_TAB_XPATH)))
            driver.execute_script("arguments[0].click();", weak_matchups_tab)
            
            print(" -> Pausing 1s after click for UI script execution...")
            time.sleep(1)
            
            print(" -> Waiting up to 8s for the top of the matchup list to render fully (div[5])...")
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located((By.XPATH, THIRD_MATCHUP_CHAMP_XPATH))
            )
            matchup_success = True
        except Exception:
            if attempt < max_attempts:
                print(" -> Matchup rendering failed. Retrying...")
                time.sleep(1) 
            else:
                print(" -> Matchup rendering failed after all attempts. Skipping.")
                
    
    if matchup_success:
        extracted_count = 0
        for i in range(1, MAX_MATCHUPS_TO_SCRAPE + 1):
            CHAMP_LINK_XPATH = f"{MATCHUP_DATA_BASE_XPATH}[{i}]/a"
            CHAMP_IMG_XPATH = f"{CHAMP_LINK_XPATH}/span/img"
            # NEW: Value location is div[3]
            WIN_RATE_XPATH = f"{MATCHUP_DATA_BASE_XPATH}[{i}]/div[{WIN_RATE_DIV_LOCATION}]" 
            
            try:
                WebDriverWait(driver, ROW_TIMEOUT).until(
                    EC.presence_of_element_located((By.XPATH, CHAMP_IMG_XPATH))
                )

                champ_img_element = driver.find_element(By.XPATH, CHAMP_IMG_XPATH)
                matchup_champ = champ_img_element.get_attribute("alt")
                value_element = driver.find_element(By.XPATH, WIN_RATE_XPATH)
                win_rate_delta = value_element.text
                
                if matchup_champ and win_rate_delta:
                    champion_data.append({
                        'champion': champion, 
                        'category': 'NORMALIZED_MATCHUP', # Updated category name
                        'vs_champ': matchup_champ, 
                        'value': win_rate_delta 
                    })
                    extracted_count += 1
                
                if i >= 2 and i % 5 == 0 and i < MAX_MATCHUPS_TO_SCRAPE: 
                    next_element_xpath = f"{MATCHUP_DATA_BASE_XPATH}[{i+1}]"
                    try:
                        WebDriverWait(driver, 1).until(
                            EC.presence_of_element_located((By.XPATH, next_element_xpath))
                        )
                        next_element = driver.find_element(By.XPATH, next_element_xpath)
                        driver.execute_script("arguments[0].scrollIntoView(false);", next_element)
                    except Exception:
                        pass

            except Exception:
                break 
        
        print(f" -> Successfully extracted {extracted_count} normalized matchup rows.")

    # --- BLOCK 2: GOOD SYNERGIES (Normalized button click & XPATH logic) ---
    print("\n[GOOD SYNERGIES - NORMALIZED]")
    try:
        # Click the NEW Normalized Synergies tab
        print(" -> Clicking Normalized Synergies tab via JS...")
        good_synergies_tab = wait.until(EC.presence_of_element_located((By.XPATH, GOOD_SYNERGIES_TAB_XPATH)))
        driver.execute_script("arguments[0].click();", good_synergies_tab)
        
        # Targeted wait after click
        time.sleep(1)

        # --- DETERMINE DATA BLOCK XPATH BASED ON CHAMPION ROLE (Logic retained) ---
        print(str(champion.title().lower()), SUPPORT_CHAMPS)
        if str(champion.title()).lower() in SUPPORT_CHAMPS: # Note: .title() is safer for checking against the set
            synergy_base_path = MATCHUP_DATA_BASE_XPATH # Use div[5] for support
            print(f" -> {champion} is a SUPPORT: Using MATCHUP_DATA_BASE_XPATH (div[5]).")
        else:
            synergy_base_path = SYNERGY_DATA_BASE_XPATH # Use div[4] for non-support
            print(f" -> {champion} is NON-SUPPORT: Using SYNERGY_DATA_BASE_XPATH (div[4]).")

        # Wait for the content to swap again using the determined base path
        FIRST_SYNERGY_CHAMP_XPATH = f"{synergy_base_path}[1]/a"
        try:
            WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, FIRST_SYNERGY_CHAMP_XPATH)))
        except TimeoutException:
            print(" -> WARNING: Synergy data failed initial load check.")


        extracted_count = 0
        for i in range(1, MAX_MATCHUPS_TO_SCRAPE + 1):
            # *** USE THE DYNAMICALLY DETERMINED synergy_base_path ***
            CHAMP_LINK_XPATH = f"{synergy_base_path}[{i}]/a"
            CHAMP_IMG_XPATH = f"{CHAMP_LINK_XPATH}/span/img"
            # NEW: Value location is div[3]
            WIN_RATE_XPATH = f"{synergy_base_path}[{i}]/div[{WIN_RATE_DIV_LOCATION}]"
            
            try:
                WebDriverWait(driver, ROW_TIMEOUT).until(
                    EC.presence_of_element_located((By.XPATH, CHAMP_IMG_XPATH))
                )

                champ_img_element = driver.find_element(By.XPATH, CHAMP_IMG_XPATH)
                synergy_champ = champ_img_element.get_attribute("alt")
                value_element = driver.find_element(By.XPATH, WIN_RATE_XPATH)
                synergy_delta = value_element.text
                
                if synergy_champ and synergy_delta:
                    champion_data.append({
                        'champion': champion, 
                        'category': 'NORMALIZED_SYNERGY', # Updated category name
                        'with_champ': synergy_champ, 
                        'value': synergy_delta 
                    })
                    extracted_count += 1

                # SCROLL OPTIMIZATION
                if i >= 2 and i % 5 == 0 and i < MAX_MATCHUPS_TO_SCRAPE:
                    next_element_xpath = f"{synergy_base_path}[{i+1}]"
                    try:
                        WebDriverWait(driver, 1).until(
                            EC.presence_of_element_located((By.XPATH, next_element_xpath))
                        )
                        next_element = driver.find_element(By.XPATH, next_element_xpath)
                        driver.execute_script("arguments[0].scrollIntoView(false);", next_element)
                    except Exception:
                        pass
                
            except Exception:
                print(f" -> Stopped extraction after {i-1} synergy rows.")
                break 
        
        print(f" -> Successfully extracted {extracted_count} normalized synergy rows.")
        
    except Exception as e:
        print(f" -> ERROR: Good Synergy scrape failed: {e}")
        
    return champion_data

# The run_scraper function remains the same.

def run_scraper():
    """Main execution function. Sets up and quits the driver for *each* champion."""
    champions = load_champions(CHAMPIONS_FILE)
    if not champions:
        return

    all_data = []

    try:
        champions_to_scrape = champions 

        for champ in champions_to_scrape:
            # 1. SETUP DRIVER
            driver = setup_driver()
            
            # 2. SCRAPE
            data = scrape_data(driver, champ)
            all_data.extend(data)
            try:
                df = pd.DataFrame(all_data)
                df.to_excel("lolalytics_data_normalized.xlsx")
            except Exception as e:
                print(f"Error! {e}")
            
            # 3. QUIT DRIVER immediately to free resources and flush cache
            driver.quit()
            
            # Small, non-critical wait between driver starts
            time.sleep(1) 
            
    except Exception as e:
        print(f"An unexpected error occurred in the main loop: {e}")

    # Convert to DataFrame and save
    if all_data:
        df = pd.DataFrame(all_data)
        df.to_excel(OUTPUT_FILE, index=False)
        print("\n--- Scraper Finished ---")
        print(f"Data successfully scraped for {len(df['champion'].unique())} champions and saved to {OUTPUT_FILE}")
        print("\nExample Data:")
        print(df.head())
    else:
        print("WARNING: No data was successfully scraped.")

if __name__ == '__main__':
    run_scraper()
    print("The scraper is now configured to extract **Normalized Win Rates**.")
    print("The tab XPATHS and the row value XPATHS have been updated to target the new data fields.")


--- Scraping Normalized Matchups and Synergies for: aatrox (Max 50 each) ---
 -> Waiting for core page content to load...
 -> Scrolling down to load dynamic content area...

[WEAK MATCHUPS - NORMALIZED]
 -> Attempt 1: Clicking Normalized Matchups tab via JS...
 -> Pausing 1s after click for UI script execution...
 -> Waiting up to 8s for the top of the matchup list to render fully (div[5])...
 -> Successfully extracted 30 normalized matchup rows.

[GOOD SYNERGIES - NORMALIZED]
 -> Clicking Normalized Synergies tab via JS...
aatrox {'tahm kench', 'nami', 'thresh', 'alistar', 'braum', 'renata glasc', 'maokai', 'brand', 'blitzcrank', 'pyke', 'rell', 'milio', 'lulu', 'taric', 'lux', 'nautilus', 'senna', 'poppy', 'pantheon', 'seraphine', 'zilean', 'zyra', 'morgana', 'velkoz', 'soraka', 'neeko', 'rakan', 'swain', 'janna', 'leona', 'karma', 'bard', 'yuumi', 'sona'}
 -> aatrox is NON-SUPPORT: Using SYNERGY_DATA_BASE_XPATH (div[4]).
 -> Stopped extraction after 30 synergy rows.
 -> Successfull

KeyboardInterrupt: 

In [18]:
import time
import pickle
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape_champion_winrates(url):
    chrome_options = Options()
    
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("window-size=1920,1080")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    driver.get(url)

    try:
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "/html/body/div[1]/div/div/div[3]/div[1]/button[2]"))
        )
        cookie_button.click()
    except Exception:
        pass

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "/html/body/main/div[6]/div[3]/div[3]/a"))
        )
    except Exception:
        driver.quit()
        return {}

    while True:
        current_champs = driver.find_elements(By.XPATH, "/html/body/main/div[6]/div[*]/div[3]/a")
        if not current_champs:
            break
            
        last_champ = current_champs[-1]
        
        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", last_champ)
        time.sleep(2)
        
        new_champs = driver.find_elements(By.XPATH, "/html/body/main/div[6]/div[*]/div[3]/a")
        if len(new_champs) == len(current_champs):
            break

    champions_data = {}
    
    name_elements = driver.find_elements(By.XPATH, "/html/body/main/div[6]/div[*]/div[3]/a")
    winrate_elements = driver.find_elements(By.XPATH, "/html/body/main/div[6]/div[*]/div[6]/div/span[1]")

    for name_el, wr_el in zip(name_elements, winrate_elements):
        name = name_el.text.strip()
        winrate = wr_el.text.strip()
        if name and winrate:
            champions_data[name] = winrate

    driver.quit()
    return champions_data

def save_to_pickle(data, filename):
    with open(filename, 'wb') as file:
        pickle.dump(data, file)

def load_from_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

if __name__ == "__main__":
    target_url = "https://lolalytics.com/lol/tierlist/?lane=bottom"
    pickle_file = "lolalytics_bot_data.pkl"
    
    scraped_results = scrape_champion_winrates(target_url)
    
    if scraped_results:
        save_to_pickle(scraped_results, pickle_file)
        
        loaded_results = load_from_pickle(pickle_file)
        
        for champ, winrate in loaded_results.items():
            print(f"{champ}: {winrate}")
    else:
        print("Data extraction failed.")

Jinx: 53.28
Katarina: 53.99
Caitlyn: 51.69
Aurelion Sol: 54.29
Samira: 53.39
Vayne: 52.67
Jhin: 51.78
Yasuo: 53.79
Kog'Maw: 54.61
Miss Fortune: 53.19
Karthus: 54.26
Kai'Sa: 50.51
Veigar: 55.40
Draven: 52.06
Ashe: 53.17
Xerath: 53.83
Viktor: 53.40
Smolder: 52.71
Sivir: 51.88
Swain: 53.71
Xayah: 53.16
Lux: 53.49
Syndra: 53.11
Nilah: 54.53
Twitch: 52.20
Vladimir: 51.52
Aphelios: 50.34
Vel'Koz: 54.49
Hwei: 54.49
Mel: 51.47
Senna: 52.93
Lucian: 50.92
Ezreal: 49.82
Brand: 54.76
Seraphine: 52.50
Yunara: 49.68
Tristana: 52.07
Cassiopeia: 51.61
Corki: 50.27
Tahm Kench: 54.96
Heimerdinger: 53.73
Ziggs: 53.29
Zeri: 51.66
Akshan: 49.10
Varus: 49.40
Kalista: 48.44
Quinn: 49.82


In [6]:
import time
import pickle
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


def scrape_champion_winrates(url):
    chrome_options = Options()

    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("window-size=1920,1080")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    driver.get(url)

    # Handle cookie consent popup
    try:
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable(
                (By.XPATH, "/html/body/div[9]/div[2]/div[2]/div[2]/div[2]/button[2]/p")
            )
        )
        cookie_button.click()
    except Exception:
        pass

    # Wait for the tier list rows to load
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.XPATH,
                 "//div[contains(@class,'rt-tbody')]//a/strong"
                 " | //div[contains(@class,'tier-list')]//a/strong")
            )
        )
    except Exception:
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a strong"))
            )
        except Exception:
            driver.quit()
            return {}

    time.sleep(3)

    # Scroll to load all rows
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(15):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(1)

    champions_data = {}

    # --- Primary extraction: class-based selectors ---
    name_elements = driver.find_elements(
        By.XPATH,
        "//div[contains(@class,'rt-tbody')]//div[contains(@class,'rt-tr')]//a/strong"
    )
    tier_elements = driver.find_elements(
        By.XPATH,
        "//div[contains(@class,'rt-tbody')]//div[contains(@class,'rt-tr')]/div[4]//span/b"
    )
    winrate_elements = driver.find_elements(
        By.XPATH,
        "//div[contains(@class,'rt-tbody')]//div[contains(@class,'rt-tr')]/div[5]//span/b"
    )

    # --- Fallback: positional XPaths based on your provided structure ---
    if not name_elements:
        base = ("/html/body/div[2]/div/div[2]/div[2]/div/div/div[1]"
                "/div/div[2]/div/div[4]/div[1]/div/div[2]/div[*]/div")

        name_elements = driver.find_elements(
            By.XPATH, f"{base}/div[3]/a/strong"
        )
        tier_elements = driver.find_elements(
            By.XPATH, f"{base}/div[4]/span/b"
        )
        winrate_elements = driver.find_elements(
            By.XPATH, f"{base}/div[5]/span/b"
        )

    for name_el, tier_el, wr_el in zip(name_elements, tier_elements, winrate_elements):
        name = name_el.text.strip()
        tier = tier_el.text.strip()
        winrate_raw = wr_el.text.strip()

        if name and winrate_raw:
            try:
                winrate = float(winrate_raw.replace("%", "").replace(",", "."))
            except ValueError:
                winrate = 0.0

            champions_data[name] = {
                "tier": tier,
                "winrate": winrate,
            }

    driver.quit()
    return champions_data


def save_to_pickle(data, filename):
    with open(filename, "wb") as file:
        pickle.dump(data, file)


def load_from_pickle(filename):
    with open(filename, "rb") as file:
        return pickle.load(file)


if __name__ == "__main__":
    target_url = "https://u.gg/lol/adc-tier-list?region=euw1&rank=emerald"
    pickle_file = "ugg_adc_data.pkl"

    scraped_results = scrape_champion_winrates(target_url)

    if scraped_results:
        save_to_pickle(scraped_results, pickle_file)

        loaded_results = load_from_pickle(pickle_file)

        for champ, data in loaded_results.items():
            print(f"{champ}: Tier {data['tier']} | Win Rate {data['winrate']:.2f}%")
    else:
        print("Data extraction failed.")

Karthus: Tier A | Win Rate 54.40%
Seraphine: Tier A | Win Rate 54.12%
Xerath: Tier A | Win Rate 53.71%
Lux: Tier A | Win Rate 53.48%
Hwei: Tier A | Win Rate 53.36%
Veigar: Tier A | Win Rate 53.00%
Vel'Koz: Tier A | Win Rate 52.97%
Ziggs: Tier A | Win Rate 52.94%
Smolder: Tier S+ | Win Rate 52.83%
Katarina: Tier A | Win Rate 52.74%
Brand: Tier S | Win Rate 52.63%
Vladimir: Tier A | Win Rate 52.21%
Aurelion Sol: Tier A | Win Rate 52.03%
Senna: Tier A | Win Rate 51.67%
Kog'Maw: Tier A | Win Rate 51.57%
Jinx: Tier S | Win Rate 51.49%
Swain: Tier A | Win Rate 51.36%
Twitch: Tier S | Win Rate 51.12%
Ashe: Tier S | Win Rate 51.05%
Nilah: Tier A | Win Rate 50.77%
Xayah: Tier B | Win Rate 50.14%
Samira: Tier B | Win Rate 49.89%
Sivir: Tier B | Win Rate 49.79%
Miss Fortune: Tier B | Win Rate 49.65%
Yasuo: Tier B | Win Rate 49.62%
Tristana: Tier B | Win Rate 49.54%
Varus: Tier B | Win Rate 49.40%
Vayne: Tier B | Win Rate 49.38%
Lucian: Tier B | Win Rate 49.22%
Zeri: Tier B | Win Rate 49.11%
Jhin:

In [7]:
pickle_file = "lolalytics_bot_data.pkl"
print(load_from_pickle(pickle_file))

{'Jinx': '53.28', 'Katarina': '53.99', 'Caitlyn': '51.69', 'Aurelion Sol': '54.29', 'Samira': '53.39', 'Vayne': '52.67', 'Jhin': '51.78', 'Yasuo': '53.79', "Kog'Maw": '54.61', 'Miss Fortune': '53.19', 'Karthus': '54.26', "Kai'Sa": '50.51', 'Veigar': '55.40', 'Draven': '52.06', 'Ashe': '53.17', 'Xerath': '53.83', 'Viktor': '53.40', 'Smolder': '52.71', 'Sivir': '51.88', 'Swain': '53.71', 'Xayah': '53.16', 'Lux': '53.49', 'Syndra': '53.11', 'Nilah': '54.53', 'Twitch': '52.20', 'Vladimir': '51.52', 'Aphelios': '50.34', "Vel'Koz": '54.49', 'Hwei': '54.49', 'Mel': '51.47', 'Senna': '52.93', 'Lucian': '50.92', 'Ezreal': '49.82', 'Brand': '54.76', 'Seraphine': '52.50', 'Yunara': '49.68', 'Tristana': '52.07', 'Cassiopeia': '51.61', 'Corki': '50.27', 'Tahm Kench': '54.96', 'Heimerdinger': '53.73', 'Ziggs': '53.29', 'Zeri': '51.66', 'Akshan': '49.10', 'Varus': '49.40', 'Kalista': '48.44', 'Quinn': '49.82'}


In [21]:
import pandas as pd
import pickle

def load_from_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

def calculate_optimal_pick(df_path, allies: list, enemies: list, good_synergy_key="GOOD_SYNERGY", matchup_key="WEAK_MATCHUP"):
    df = pd.read_excel(df_path)

    """
    Calculates the combined Win Rate Delta for every champion against a specific draft.
    The highest score indicates the most optimal pick.
    """
    
    try:
        df['value_numeric'] = pd.to_numeric(df['value'], errors='coerce')
        # Convert win rate (e.g., 51.86) to delta (e.g., +1.86)
        df['delta'] = df['value_numeric'] - 50.0
        df.dropna(subset=['delta'], inplace=True)
    except Exception as e:
        print(f"Error converting 'value' column to numeric: {e}. Check your data format.")
        return None

    # Rename columns for consistent merging/grouping
    # NORMALIZED_MATCHUP
    df['matchup_champ'] = df['vs_champ'].fillna(df['with_champ'])

    enemy_matchups = df[
        (df['category'] == matchup_key) & (df['champion'].isin(enemies))
    ].copy()
    print(f"missing champs: {set(enemies)-set(enemy_matchups['champion'])}")
    
    # Champions where the selected pick is an ally (GOOD_SYNERGY)
    # NORMALIZED_SYNERGY
    ally_synergies = df[
        (df['category'] == good_synergy_key) & (df['champion'].isin(allies))
    ].copy()
    print(f"missing champs: {set(allies)-set(ally_synergies['champion'])}")

    matchup_dict = {}
    winrates_dict = {}
    if good_synergy_key=="GOOD_SYNERGY":
        percent_base = 100
        percent_base2 = 0
    else:
        percent_base = 50
        percent_base2 = 50

    for index, enemy_matchup in enemy_matchups.iterrows():
        if enemy_matchup["vs_champ"] in matchup_dict.keys():
            matchup_dict[enemy_matchup["vs_champ"]].append(percent_base-enemy_matchup["value"])
            # if str(enemy_matchup["vs_champ"]) == "Xayah":
            #     print(enemy_matchup)
            #     print(enemy_matchup["vs_champ"], percent_base-enemy_matchup["value"])
            #     print(matchup_dict[enemy_matchup["vs_champ"]])
        else:
            matchup_dict[enemy_matchup["vs_champ"]] = [percent_base-enemy_matchup["value"]]
    # print(matchup_dict["Xayah"])
    for index, synergy in ally_synergies.iterrows():
        if synergy["with_champ"] in matchup_dict.keys():
            matchup_dict[synergy["with_champ"]].append(percent_base2+synergy["value"])
            # if str(synergy["with_champ"]) == "Xayah":
                # print(synergy)
                # print(synergy["with_champ"], percent_base2+synergy["value"])
                # print(matchup_dict[synergy["with_champ"]])
        else:
            matchup_dict[synergy["with_champ"]] = [percent_base2+synergy["value"]]
    # print(matchup_dict["Xayah"])
    # print(f"matchup_dict: {matchup_dict}")
    max_len = max([len(value) for value in matchup_dict.values()])
    for key, values in matchup_dict.items():
        if len(values) == max_len:
            winrates_dict[key] = round(sum(values) / len(values), 2)

    optimal_champs = {key: value for key, value in winrates_dict.items() if key in adcs}
    sorted_optimal_champs = dict(sorted(optimal_champs.items(), key=lambda item: item[1], reverse=True))
    # print(sorted_optimal_champs)
    return sorted_optimal_champs

adcs = [
    "Aphelios",
    "Ashe",
    "Caitlyn",
    "Corki",
    "Draven",
    "Ezreal",
    "Jhin",
    "Jinx",
    "Kai'Sa",
    "Kalista",
    "Kindred",
    "Kog'Maw",
    "Lucian",
    "Miss Fortune",
    "Nilah",
    "Quinn",
    "Samira",
    "Senna",
    "Sivir",
    "Smolder",
    "Tristana",
    "Twitch",
    "Varus",
    "Vayne",
    "Xayah",
    "Zeri",
    "Swain",
    "Seraphine",
    "Mel",
    "Yunara", 
    "Ziggs",
    "Kathus",
    "Xerath",
    "Veigar",
    "Yasuo"
]
allied_champs = ["nasus","mel","",""]
enemy_champs =  ["garen","twitch","viktor","",""]

enemy_champs = [enemy_champ.lower() for enemy_champ in enemy_champs]
allied_champs = [allied_champs.lower() for allied_champs in allied_champs]
# sorted_winrate_champs = calculate_optimal_pick("lolalytics_data_normalized_percent.xlsx", allied_champs, enemy_champs, good_synergy_key="NORMALIZED_SYNERGY", matchup_key="NORMALIZED_MATCHUP")
# pickle_file = "lolalytics_bot_data.pkl"
pickle_file = "ugg_adc_data.pkl"
sorted_winrate_champs = load_from_pickle(pickle_file)
sorted_winrate_champs = {key: float(value['winrate']) for key, value in sorted_winrate_champs.items()}
print(f"U.GG Winrates percent: {sorted_winrate_champs}")
# sorted_winrate_champs = {}

# sorted_optimal_champs = calculate_optimal_pick("lolalytics_data_new.xlsx", allied_champs, enemy_champs, 
#                                         good_synergy_key="NORMALIZED_SYNERGY", matchup_key="NORMALIZED_MATCHUP",)
sorted_optimal_champs = calculate_optimal_pick("lolalytics_data_percent.xlsx", allied_champs, enemy_champs, 
                                        good_synergy_key="GOOD_SYNERGY", matchup_key="WEAK_MATCHUP",)
print(f"Winrates delta: {sorted_optimal_champs}")

own_winrates = {"Aphelios": (40+5)/(78+11)*100, "Samira": (23+6)/(41+10)*100, "Ezreal": 19/(36+2)*100, "Caitlyn": 12/25*100, 
                "Ashe": (20+6)/(31+17)*100, "Kog'Maw": (8+2)/(18+7)*100, "Jhin": (5+6)/(9+10)*100, 
                "Nilah": 8/20*100, "Sivir": 21/28*100} # , "Smolder": 7/15*100, "Lucian": (7+2)/(15+3)*100
# print(f"Own winrates: {sorted(own_winrates.items(), key=lambda item: item[1])}")
own_winrate = 0.1
pecent_winrate = 0.00
delta_winrate = 0.9
delta_winrate2 = 1.0
own_winrate_optimal_champs = {}
for champ, winrate in sorted_optimal_champs.items():
    if champ in own_winrates.keys() and champ in sorted_winrate_champs.keys():
        own_winrate_optimal_champs[champ] = round((sorted_winrate_champs[champ]*pecent_winrate+own_winrates[champ]*own_winrate+sorted_optimal_champs[champ]*delta_winrate), 2)
    elif champ in own_winrates.keys():
        own_winrate_optimal_champs[champ] = round((sorted_optimal_champs[champ]*delta_winrate2)+own_winrates[champ]*own_winrate, 2)
    elif champ in sorted_winrate_champs.keys():
        own_winrate_optimal_champs[champ] = round(sorted_optimal_champs[champ]*delta_winrate2+sorted_winrate_champs[champ]*pecent_winrate, 2)
    else:
        own_winrate_optimal_champs[champ] = round(sorted_optimal_champs[champ], 2)

sorted_own_winrate = dict(sorted(own_winrate_optimal_champs.items(), key=lambda item: item[1], reverse=True))

print(f"Own winrates with delta: {sorted_own_winrate}")


U.GG Winrates percent: {'Karthus': 54.4, 'Seraphine': 54.12, 'Xerath': 53.71, 'Lux': 53.48, 'Hwei': 53.36, 'Veigar': 53.0, "Vel'Koz": 52.97, 'Ziggs': 52.94, 'Smolder': 52.83, 'Katarina': 52.74, 'Brand': 52.63, 'Vladimir': 52.21, 'Aurelion Sol': 52.03, 'Senna': 51.67, "Kog'Maw": 51.57, 'Jinx': 51.49, 'Swain': 51.36, 'Twitch': 51.12, 'Ashe': 51.05, 'Nilah': 50.77, 'Xayah': 50.14, 'Samira': 49.89, 'Sivir': 49.79, 'Miss Fortune': 49.65, 'Yasuo': 49.62, 'Tristana': 49.54, 'Varus': 49.4, 'Vayne': 49.38, 'Lucian': 49.22, 'Zeri': 49.11, 'Jhin': 49.07, 'Caitlyn': 49.05, 'Aphelios': 48.99, 'Draven': 48.9, 'Mel': 47.95, 'Kalista': 47.94, 'Ezreal': 47.06, 'Yunara': 46.99, "Kai'Sa": 46.75, 'Corki': 46.29}
missing champs: {''}
missing champs: {''}
Winrates delta: {'Senna': 52.84, 'Swain': 52.79, 'Nilah': 51.24, 'Xayah': 50.96, 'Tristana': 50.22, 'Draven': 50.21, 'Jinx': 49.17, 'Vayne': 48.5, 'Samira': 48.32, 'Ashe': 48.24, 'Yunara': 48.03, 'Zeri': 47.89, 'Ziggs': 47.46, 'Jhin': 47.37, 'Caitlyn': 47.